<a href="https://colab.research.google.com/github/tirthankarbiswas24/masai_quickpay/blob/main/04_python/fintech_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


Loading the files ledger.csv and gateway.csv
*   Explore the data.
*   Check nulls
*   Check duplicates




In [4]:
#ledger_df = pd.read_csv("../01_data/raw/ledger.csv")
#gateway_df = pd.read_csv("./01_data/raw/gateway.csv")
ledger_df = pd.read_csv("/content/drive/Othercomputers/My Laptop/BITSOM/Module-2/ledger.csv")
gateway_df = pd.read_csv("/content/drive/Othercomputers/My Laptop/BITSOM/Module-2/gateway.csv")
print(ledger_df.shape)
print(gateway_df.shape)
#print(ledger_df.describe())

(10, 6)
(9, 6)


In [6]:
# Cleaning the data frames loaded
# Checking nulls and duplicate transaction IDs in gateway and ledger
for df in [ledger_df, gateway_df]:
    df.columns = df.columns.str.strip().str.lower()
#print(ledger_df.columns)
#print(gateway_df.columns)

ledger_duplicates = ledger_df['transaction_id'].duplicated().sum()
gateway_duplicates = gateway_df['transaction_id'].duplicated().sum()
print("ledger_duplicates: ", ledger_duplicates, "gateway_duplicates: ", gateway_duplicates)

#unique_count_ledger = ledger_df['transaction_id'].nunique()
#count_ledger = ledger_df['transaction_id'].count()
#unique_count_gateway = gateway_df['transaction_id'].nunique()
#count_gateway = gateway_df['transaction_id'].count()
#if unique_count_ledger == count_ledger:
#  print("There are no duplicate ledger records. There are ", unique_count_ledger, " unique records and total count of records is also ", count_ledger)
#else:
#  print("There are dupilcate ledger records. There are ", unique_count_ledger, " unique records and total count of records is ", count_ledger)
#if unique_count_gateway == count_gateway:
#  print("There are no duplicate gateway records. There are ", unique_count_ledger, " unique records and total count of records is also ", count_ledger)
#else:
#  print("There are dupilcate gateway records. There are ", unique_count_ledger, " unique records and total count of records is ", count_ledger)

ledger_null = ledger_df.isnull().sum().sum()
gateway_null = gateway_df.isnull().sum().sum()

if ledger_null == 0 and gateway_null == 0:
  print("There no nulls in gateway or ledger")
else:
  print("There are ", ledger_null, " null in ledger and ", gateway_null, " null in gateway")


ledger_duplicates:  0 gateway_duplicates:  0
There no nulls in gateway or ledger


Identifying missing records:
*   Identify records missing in gateway
*   Identify records missing in ledger


In [7]:
ledger_ids = set(ledger_df['transaction_id'])
gateway_ids = set(gateway_df['transaction_id'])
#print("Ledger ids: ", len(ledger_ids), ledger_ids)
#print("gateway ids: ", len(gateway_ids), gateway_ids)
missing_in_gateway_ids = sorted(ledger_ids - gateway_ids)
missing_in_ledger_ids = sorted(gateway_ids - ledger_ids)
print("missing in gateway: ", missing_in_gateway_ids, "missing in ledger: ", missing_in_ledger_ids)
missing_in_gateway_df = ledger_df[ledger_df['transaction_id'].isin(missing_in_gateway_ids)].copy()
missing_in_gateway_df['issue'] = 'MISSING_IN_GATEWAY'
print("missing_in_gateway: ", missing_in_gateway_df.head())
missing_in_ledger_df = gateway_df[gateway_df['transaction_id'].isin(missing_in_ledger_ids)].copy()
missing_in_ledger_df['issue'] = 'MISSING_IN_LEDGER'
print("missing_in_ledger: ", missing_in_ledger_df.head())


missing in gateway:  ['R004', 'R010'] missing in ledger:  ['R011']
missing_in_gateway:    transaction_id transaction_date merchant_id  amount_usd   status  \
3           R004       2026-03-02        M003      2100.0  success   
9           R010       2026-03-05        M004      2500.0  success   

  payment_method               issue  
3           Card  MISSING_IN_GATEWAY  
9         Wallet  MISSING_IN_GATEWAY  
missing_in_ledger:    transaction_id transaction_date merchant_id  amount_usd   status  \
8           R011       2026-03-05        M003      1800.0  success   

  payment_method              issue  
8           Card  MISSING_IN_LEDGER  


Identify amount mismatches

In [14]:
common_ids = ledger_ids & gateway_ids
print("common IDs: ", common_ids)
common_ledger_df = ledger_df[ledger_df['transaction_id'].isin(common_ids)]
common_gateway_df = gateway_df[gateway_df['transaction_id'].isin(common_ids)]
comparison_df = pd.merge(common_ledger_df, common_gateway_df, on='transaction_id',
                         suffixes=("_gateway", "_ledger"), )
amount_mismatch_df = comparison_df[comparison_df['amount_usd_gateway'] != comparison_df['amount_usd_ledger']].copy()
amount_mismatch_df['amount_mismatch'] = comparison_df['amount_usd_ledger'] - comparison_df['amount_usd_gateway']
print(len(amount_mismatch_df), amount_mismatch_df[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'amount_mismatch']])
print(amount_mismatch_df.columns)

common IDs:  {'R005', 'R002', 'R009', 'R006', 'R007', 'R003', 'R001', 'R008'}
2   transaction_id  amount_usd_ledger  amount_usd_gateway  amount_mismatch
1           R002              900.0               850.0             50.0
6           R008              600.0               640.0            -40.0
Index(['transaction_id', 'transaction_date_gateway', 'merchant_id_gateway',
       'amount_usd_gateway', 'status_gateway', 'payment_method_gateway',
       'transaction_date_ledger', 'merchant_id_ledger', 'amount_usd_ledger',
       'status_ledger', 'payment_method_ledger', 'amount_mismatch'],
      dtype='object')


Identify status mismatches

In [18]:
status_mismatch_df = comparison_df[comparison_df['status_ledger'] != comparison_df['status_gateway']].copy()
print(len(status_mismatch_df), status_mismatch_df[['transaction_id', 'status_ledger', 'status_gateway']])
print(status_mismatch_df.columns)

1   transaction_id status_ledger status_gateway
3           R005        failed        success
Index(['transaction_id', 'transaction_date_gateway', 'merchant_id_gateway',
       'amount_usd_gateway', 'status_gateway', 'payment_method_gateway',
       'transaction_date_ledger', 'merchant_id_ledger', 'amount_usd_ledger',
       'status_ledger', 'payment_method_ledger'],
      dtype='object')


Build the final reconciliation report

In [24]:
missing_in_gateway_report_df = missing_in_gateway_df[['transaction_id', 'transaction_date', 'amount_usd']].copy()
missing_in_gateway_report_df['amount_usd_gateway'] = 0
missing_in_gateway_report_df['amount_mismatch'] = missing_in_gateway_report_df['amount_usd']
missing_in_gateway_report_df['status_ledger'] = missing_in_gateway_df['status']
missing_in_gateway_report_df['status_gateway'] = "NA"
missing_in_gateway_report_df['issue'] = missing_in_gateway_df['issue']
missing_in_gateway_report_df.columns=['transaction_id', 'transaction_date', 'amount_usd_ledger', 'amount_usd_gateway', 'amount_mismatch', 'status_ledger', 'status_gateway', 'issue']
print("missing_in_gateway_report_df: ", missing_in_gateway_report_df.head(2))

missing_in_ledger_report_df = missing_in_ledger_df[['transaction_id', 'transaction_date']].copy()
missing_in_ledger_report_df['amount_usd_ledger'] = 0
missing_in_ledger_report_df['amount_usd_gateway'] = missing_in_ledger_df['amount_usd']
missing_in_ledger_report_df['amount_mismatch'] = -missing_in_ledger_report_df['amount_usd_gateway']
missing_in_ledger_report_df['status_ledger'] = "NA"
missing_in_ledger_report_df['status_gateway'] = missing_in_ledger_df['status']
missing_in_ledger_report_df['issue'] =  missing_in_ledger_df['issue']
print("missing_in_ledger_report_df: ", missing_in_ledger_report_df.head())

amount_mismatch_report_df = amount_mismatch_df[['transaction_id', 'transaction_date_gateway', 'amount_usd_ledger', 'amount_usd_gateway',
                                                'amount_mismatch', 'status_ledger', 'status_gateway']].copy()
amount_mismatch_report_df['issue'] = "AMOUNT_MISMATCH"
amount_mismatch_report_df.columns=['transaction_id', 'transaction_date', 'amount_usd_ledger', 'amount_usd_gateway', 'amount_mismatch', 'status_ledger', 'status_gateway', 'issue']
print("amount_mismatch: ", amount_mismatch_report_df.head())

status_mismatch_report_df = status_mismatch_df[['transaction_id', 'transaction_date_gateway', 'amount_usd_ledger', 'amount_usd_gateway']].copy()
status_mismatch_report_df['amount_mismatch'] = status_mismatch_df['amount_usd_ledger'] - status_mismatch_df['amount_usd_gateway']
status_mismatch_report_df[['status_ledger', 'status_gateway']] = status_mismatch_df[['status_ledger', 'status_gateway']]
status_mismatch_report_df['issue'] = "STATUS_MISMATCH"
status_mismatch_report_df.columns = ['transaction_id', 'transaction_date', 'amount_usd_ledger', 'amount_usd_gateway', 'amount_mismatch', 'status_ledger', 'status_gateway', 'issue']
print("status mismatch: ", status_mismatch_report_df.head())


missing_in_gateway_report_df:    transaction_id transaction_date  amount_usd_ledger  amount_usd_gateway  \
3           R004       2026-03-02             2100.0                   0   
9           R010       2026-03-05             2500.0                   0   

   amount_mismatch status_ledger status_gateway               issue  
3           2100.0       success             NA  MISSING_IN_GATEWAY  
9           2500.0       success             NA  MISSING_IN_GATEWAY  
missing_in_ledger_report_df:    transaction_id transaction_date  amount_usd_ledger  amount_usd_gateway  \
8           R011       2026-03-05                  0              1800.0   

   amount_mismatch status_ledger status_gateway              issue  
8          -1800.0            NA        success  MISSING_IN_LEDGER  
amount_mismatch:    transaction_id transaction_date  amount_usd_ledger  amount_usd_gateway  \
1           R002       2026-03-01              900.0               850.0   
6           R008       2026-03-04      

Generate summary metrics